In [0]:
# lab_powerpath_raw_ingest
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
import xlrd

VOLUME = "/Volumes/opsanalytics_adb_workspace01/lab/raw_data/power_path"
RAW    = "opsanalytics_adb_workspace01.lab.powerpath_raw"

In [0]:
# --- find files not yet ingested -------------------------------------------
all_files = [f.path for f in dbutils.fs.ls(VOLUME) if f.name.endswith((".xlsx", ".xls"))]

try:
    done = {r._source_file for r in spark.table(RAW).select("_source_file").distinct().collect()}
except AnalysisException:
    done = set()                      # first run, table doesn't exist yet

pending = sorted(f for f in all_files if f.split("/")[-1] not in done)
print(f"{len(pending)} file(s) to ingest")

# --- land each file as-is ---------------------------------------------------
for path in pending:
    local = path.replace("dbfs:/Volumes", "/Volumes")
    pdf = pd.read_excel(local, sheet_name=0, skiprows=1, dtype=str)
    pdf = pdf.iloc[:-1]                                    # drop trailing footer row
    pdf.columns = [str(c).strip().replace(" ", "_").replace(".", "_") for c in pdf.columns]
    pdf = pdf.loc[:, [c for c in pdf.columns if not c.startswith("Unnamed")]]

    if pdf.empty:
        print(f"SKIP (empty): {path}")
        continue

    sdf = (spark.createDataFrame(pdf)
           .withColumn("_source_file", F.lit(path.split("/")[-1]))
           .withColumn("_ingested_at", F.current_timestamp()))

    (sdf.write
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(RAW))

    print(f"OK ({sdf.count()} rows): {path}")

In [0]:
print(spark.table(RAW).select("_source_file").distinct().collect())